In [1]:
import os
for r, d, f in os.walk('/kaggle/input/notebooks/rayyanshuda/03-pytorch-and-baselines'):
    print(r, '->', f)

/kaggle/input/notebooks/rayyanshuda/03-pytorch-and-baselines -> ['evaluate.py', '__results__.html', '__notebook__.ipynb', '__output__.json', 'results.csv', 'custom.css']
/kaggle/input/notebooks/rayyanshuda/03-pytorch-and-baselines/__pycache__ -> ['evaluate.cpython-312.pyc']
/kaggle/input/notebooks/rayyanshuda/03-pytorch-and-baselines/__results___files -> ['__results___5_0.png']


In [2]:
import sys
sys.path.insert(0, '/kaggle/input/notebooks/rayyanshuda/03-pytorch-and-baselines')
from evaluate import compute_metrics, evaluate_all

In [3]:
from torchvision import transforms as T

MEAN, STD, SIZE = [0.485, 0.456, 0.406], [0.229, 0.224, 0.225], 160

train_tf = T.Compose([
    T.RandomResizedCrop(SIZE, scale=(0.6, 1.0)),
    T.RandomHorizontalFlip(),
    T.ToTensor(),
    T.Normalize(MEAN, STD),
])

eval_tf = T.Compose([
    T.Resize(SIZE),
    T.CenterCrop(SIZE),
    T.ToTensor(),
    T.Normalize(MEAN, STD),
])

In [4]:
import os, torch, pandas as pd
from PIL import Image
from torch.utils.data import Dataset, DataLoader

print(os.listdir('/kaggle/input/notebooks/rayyanshuda/02-split-and-preprocess'))          # find your notebook-output folder name first

IN   = '/kaggle/input/notebooks/rayyanshuda/02-split-and-preprocess'
CSV  = f'{IN}/split_v1.csv'
IMGS = f'{IN}/resized'

class WildfireDataset(Dataset):
    def __init__(self, csv_path, img_dir, split, transform):
        # read the csv, keep only rows where split == this split
        df = pd.read_csv(csv_path)
        df = df[df.split == split].reset_index(drop=True)
        # TODO store what you need: the out_name list and the label list
        # label: fire -> 1.0, nofire -> 0.0   (float, for BCEWithLogitsLoss)
        self.files = df.out_name.tolist()
        self.labels = (df.cls == 'fire').astype('float32').tolist()
        # TODO store img_dir and transform
        self.img_dir = img_dir
        self.transform = transform

    def __len__(self):
        # how many examples in this split?
        return len(self.files) # DataLoader uses this to know the range of i

    def __getitem__(self, i):
        # open the image at img_dir/out_name[i], convert to RGB
        path = os.path.join(self.img_dir, self.files[i])
        img = Image.open(path).convert('RGB') # RGBA -> RGB, else 4-channel tensors
        # apply self.transform
        img = self.transform(img) # PIL -> augmented, normalized tensor
        label = torch.tensor(self.labels[i], dtype=torch.float32) # float, for BCEWithLogitsLoss
        return img, label

print(len(WildfireDataset(CSV, IMGS, 'train', train_tf)),
      len(WildfireDataset(CSV, IMGS, 'val',   eval_tf)),
      len(WildfireDataset(CSV, IMGS, 'test',  eval_tf)))    # expect 1803 420 476

['__results__.html', 'resized', 'inventory.csv', '__notebook__.ipynb', '__output__.json', 'split_v1.csv', 'custom.css']
1803 420 476


In [5]:
from torchvision.models import efficientnet_b0, EfficientNet_B0_Weights
import torch.nn as nn

class ModelB(nn.Module):
    def __init__(self, pretrained=True, dropout=0.3):
        super().__init__()
        weights = EfficientNet_B0_Weights.DEFAULT if pretrained else None
        self.net = efficientnet_b0(weights=weights)
        self.net.classifier = nn.Sequential(          # replace the 1000-class ImageNet head
            nn.Dropout(dropout),
            nn.Linear(1280, 1),
        )

    def forward(self, x):
        return self.net(x).squeeze(1)

    def backbone_parameters(self):
        return self.net.features.parameters()

    def head_parameters(self):
        return self.net.classifier.parameters()

In [6]:
import copy, numpy as np, pandas as pd, torch, torch.nn as nn
from torch.utils.data import DataLoader, Subset
from sklearn.metrics import roc_auc_score, roc_curve, log_loss


def predict(model, loader):
    """Return (y_true, y_score) as numpy. The only torch-aware piece."""
    model.eval()
    ys, ss = [], []
    with torch.no_grad():
        for xb, yb in loader:
            logits = model(xb.to(DEVICE))
            ss.append(torch.sigmoid(logits).cpu().numpy())
            ys.append(yb.numpy())
    return np.concatenate(ys), np.concatenate(ss)


def pick_threshold(y_true, y_score):
    """Youden's J: maximise (sensitivity + specificity - 1). Chosen on VAL only."""
    fpr, tpr, thr = roc_curve(y_true, y_score)
    t = float(thr[np.argmax(tpr - fpr)])
    return t if np.isfinite(t) else 0.5

def train_model(make_model, seed, epochs=40, lr=1e-3, patience=8, bs=32,
                tag='model', limit=None, freeze_epochs=0):
    torch.manual_seed(seed); np.random.seed(seed)

    train_ds = WildfireDataset(CSV, IMGS, 'train', train_tf)
    val_ds   = WildfireDataset(CSV, IMGS, 'val',   eval_tf)
    if limit:
        g = torch.Generator().manual_seed(0)
        train_ds = Subset(train_ds, torch.randperm(len(train_ds), generator=g)[:limit].tolist())
        val_ds   = Subset(val_ds,   torch.randperm(len(val_ds),   generator=g)[:limit].tolist())

    train_dl = DataLoader(train_ds, batch_size=bs, shuffle=True, num_workers=2,
                          pin_memory=True, drop_last=True)
    val_dl   = DataLoader(val_ds, batch_size=64, shuffle=False, num_workers=2, pin_memory=True)

    model     = make_model().to(DEVICE)
    criterion = nn.BCEWithLogitsLoss()

    def build_optim(phase):
        if phase == 'frozen':
            for p in model.backbone_parameters(): p.requires_grad = False
            opt = torch.optim.Adam(model.head_parameters(), lr=lr)
            T   = freeze_epochs
        elif phase == 'unfrozen':
            for p in model.backbone_parameters(): p.requires_grad = True
            opt = torch.optim.Adam([
                {'params': model.backbone_parameters(), 'lr': lr / 100},   # nudge, don't relocate
                {'params': model.head_parameters(),     'lr': lr / 10},
            ])
            T   = epochs - freeze_epochs
        else:                                                  # single-phase = Model A
            opt = torch.optim.Adam(model.parameters(), lr=lr)
            T   = epochs
        return opt, torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=T)

    optimizer, scheduler = build_optim('frozen' if freeze_epochs > 0 else 'single')

    best_auc, best_state, best_epoch, bad = -1, None, -1, 0
    history = []

    for epoch in range(epochs):
        if freeze_epochs > 0 and epoch == freeze_epochs:
            optimizer, scheduler = build_optim('unfrozen')
            print('  --- backbone unfrozen ---')

        model.train()
        run_loss, n = 0.0, 0
        for xb, yb in train_dl:
            xb, yb = xb.to(DEVICE), yb.to(DEVICE)
            optimizer.zero_grad()
            loss = criterion(model(xb), yb)
            loss.backward()
            optimizer.step()
            run_loss += loss.item() * len(yb); n += len(yb)
        scheduler.step()

        yt, ys   = predict(model, val_dl)
        val_loss = log_loss(yt, np.clip(ys, 1e-7, 1 - 1e-7), labels=[0, 1])
        val_auc  = roc_auc_score(yt, ys)

        history.append({'epoch': epoch, 'train_loss': run_loss / n, 'val_loss': val_loss,
                        'val_auc': val_auc, 'lr': scheduler.get_last_lr()[0]})
        print(f'  ep {epoch:2d}  train {run_loss/n:.4f}  val {val_loss:.4f}  auc {val_auc:.4f}')

        if val_auc > best_auc:
            best_auc, best_epoch, bad = val_auc, epoch, 0
            best_state = copy.deepcopy(model.state_dict())
        else:
            bad += 1
            if bad >= patience:
                print(f'  early stop at {epoch}; best epoch was {best_epoch}')
                break

    model.load_state_dict(best_state)
    torch.save(best_state, f'/kaggle/working/{tag}_seed{seed}.pt')
    pd.DataFrame(history).to_csv(f'/kaggle/working/{tag}_seed{seed}_history.csv', index=False)
    yt_v, ys_v = predict(model, val_dl)
    return model, pd.DataFrame(history), best_auc, best_epoch, pick_threshold(yt_v, ys_v)

In [7]:
import torch

# Define the missing DEVICE variable

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

In [8]:
import torch
import numpy as np

test_dl = DataLoader(WildfireDataset(CSV, IMGS, 'test', eval_tf),
                     batch_size=64, shuffle=False, num_workers=2, pin_memory=True)

test_meta = pd.read_csv(CSV)
test_meta = test_meta[test_meta.split == 'test'].reset_index(drop=True)
print(len(test_meta), 'test rows |', test_meta.src.value_counts().to_dict())

DRY = False                     # <<< set False for the real run

EPOCHS = 4    if DRY else 40
LIMIT  = 128  if DRY else None
SEEDS  = [0]  if DRY else [0, 1, 2]

ARMS = [
    ('model_b_pretrained', lambda: ModelB(pretrained=True),  3),
    ('model_b_scratch',    lambda: ModelB(pretrained=False), 0),
]

all_rows = []
for tag, maker, fz in ARMS:
    for seed in SEEDS:
        print(f'===== {tag} seed {seed} =====')
        model, hist, bauc, bep, thr = train_model(
            maker, seed, epochs=EPOCHS, tag=tag, limit=LIMIT,
            freeze_epochs=(min(fz, 1) if DRY else fz))
        print(f'  best val auc {bauc:.4f} @ epoch {bep} | threshold {thr:.3f}')
        yt, ys = predict(model, test_dl)
        all_rows.append(evaluate_all(yt, ys, test_meta.src.values,
                                     f'{tag}_seed{seed}', threshold=thr))

results = pd.concat(all_rows, ignore_index=True)
results.to_csv('/kaggle/working/model_b_results.csv', index=False)

print('\n===== mean across seeds =====')
print(results.groupby(['model', 'slice'])[['accuracy', 'roc_auc', 'recall', 'specificity']]
      .mean().round(4).to_string())

476 test rows | {'flickr': 247, 'unsplash': 229}
===== model_b_pretrained seed 0 =====
Downloading: "https://download.pytorch.org/models/efficientnet_b0_rwightman-7f5810bc.pth" to /root/.cache/torch/hub/checkpoints/efficientnet_b0_rwightman-7f5810bc.pth


100%|██████████| 20.5M/20.5M [00:00<00:00, 134MB/s] 


  ep  0  train 0.5783  val 0.4979  auc 0.8764
  ep  1  train 0.4606  val 0.4442  auc 0.8853
  ep  2  train 0.4203  val 0.4244  auc 0.9010
  --- backbone unfrozen ---
  ep  3  train 0.4142  val 0.3882  auc 0.9222
  ep  4  train 0.3791  val 0.3516  auc 0.9358
  ep  5  train 0.3606  val 0.3337  auc 0.9461
  ep  6  train 0.3128  val 0.3064  auc 0.9501
  ep  7  train 0.2937  val 0.2822  auc 0.9571
  ep  8  train 0.2820  val 0.2657  auc 0.9623
  ep  9  train 0.2575  val 0.2564  auc 0.9641
  ep 10  train 0.2397  val 0.2464  auc 0.9664
  ep 11  train 0.2165  val 0.2336  auc 0.9684
  ep 12  train 0.2040  val 0.2223  auc 0.9708
  ep 13  train 0.2066  val 0.2167  auc 0.9729
  ep 14  train 0.1932  val 0.2133  auc 0.9741
  ep 15  train 0.1831  val 0.2088  auc 0.9743
  ep 16  train 0.1795  val 0.2013  auc 0.9762
  ep 17  train 0.1650  val 0.1995  auc 0.9761
  ep 18  train 0.1642  val 0.1933  auc 0.9776
  ep 19  train 0.1680  val 0.1951  auc 0.9768
  ep 20  train 0.1572  val 0.1895  auc 0.9782
  ep 2

In [9]:
a, b = ModelB(pretrained=True), ModelB(pretrained=True)
c, d = ModelB(pretrained=False), ModelB(pretrained=False)
same = lambda m, n: torch.allclose(next(m.net.features.parameters()),
                                   next(n.net.features.parameters()))
print('two pretrained identical:', same(a, b))    # must be True  (loaded from file)
print('two random differ:      ', not same(c, d)) # must be True  (fresh init each time)
print('params:', sum(p.numel() for p in a.parameters()))   # 4,008,829

two pretrained identical: True
two random differ:       True
params: 4008829
